# GPU virtual try-on: real person, real uploaded garment

This notebook contains the current project inference source and exact retained
striped-shirt regression images. It uses the pinned pretrained **CatVTON
mix-48k-1024** checkpoint at **768 × 1024**, 50 steps, guidance 2.5, seed 42,
with the FASHN human parser and identity-preserving composition.

The earlier local output lost the stripes and was rejected. More GPU memory
permits the larger checkpoint and resolution; it does **not** establish
image quality. Every generated image below requires visual inspection.
A rejected model output is shown only as a labeled diagnostic, never a success.
No pasted-garment fallback is used. The independent brand size engine remains
in the application; this inference notebook does not invent body measurements
or sizes.

Use a GPU runtime. The supplied cells install into a separate Python
environment without replacing Colab's preinstalled packages. The notebook
contains no credentials, private upload directory, database, or model weights.
It downloads pinned model assets from their official hosts. Google Drive is
not accessed unless the optional persistence cell is explicitly enabled.

Research use only: bundled upstream code/models retain their own licenses
(including CatVTON CC-BY-NC-SA-4.0 and the FASHN parser's license).
The five example photos are unpaired evaluation fixtures, **not** a valid
training/fine-tuning dataset. No training is launched by this notebook.

In [ ]:
import base64
from pathlib import Path

def extract_verified_source(payload, expected_sha256, destination):
    import hashlib
    import io
    import json
    import stat
    import zipfile
    from pathlib import Path, PurePosixPath

    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise ValueError("Embedded archive checksum mismatch")
    destination = Path(destination).resolve()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        entries = archive.infolist()
        names = [entry.filename for entry in entries]
        if len(names) != len(set(names)) or len(entries) > 100:
            raise ValueError("Duplicate or excessive archive entries")
        if sum(entry.file_size for entry in entries) > 12 * 1024 * 1024:
            raise ValueError("Expanded archive is unexpectedly large")
        for entry in entries:
            name = entry.orig_filename
            path = PurePosixPath(name)
            mode = entry.external_attr >> 16
            if (not name or path.is_absolute() or "\\" in name or ":" in name
                    or any(part in {"", ".", ".."} for part in name.split("/"))
                    or name != path.as_posix() or entry.is_dir()
                    or (mode and not stat.S_ISREG(mode))):
                raise ValueError(f"Unsafe archive member: {name!r}")
            target = destination / name
            if target.is_symlink() or not target.resolve().is_relative_to(destination):
                raise ValueError(f"Unsafe extraction destination: {name!r}")
        manifest = json.loads(archive.read("SOURCE_MANIFEST.json"))
        expected_names = set(manifest["files"]) | {"SOURCE_MANIFEST.json"}
        if set(names) != expected_names:
            raise ValueError("Archive contents disagree with source manifest")
        verified = {}
        for name, expected in manifest["files"].items():
            data = archive.read(name)
            if (len(data) != expected["bytes"]
                    or hashlib.sha256(data).hexdigest() != expected["sha256"]):
                raise ValueError(f"Source checksum mismatch: {name}")
            verified[name] = data
        verified["SOURCE_MANIFEST.json"] = archive.read("SOURCE_MANIFEST.json")
    destination.mkdir(parents=True, exist_ok=True)
    for name, data in verified.items():
        target = destination / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
    return manifest

SOURCE_SHA256 = "276c40f108db2ba570a85275309472284c81708c6dfb5c38263ba4db3a6da06e"
EMBEDDED_SOURCE = "..."
PROJECT = Path("/content") / ("avatar-vton-" + SOURCE_SHA256[:12])
SOURCE_MANIFEST = extract_verified_source(
    base64.b64decode(EMBEDDED_SOURCE, validate=True), SOURCE_SHA256, PROJECT
)
del EMBEDDED_SOURCE
print(f"Verified {len(SOURCE_MANIFEST['files'])} source files in {PROJECT}")

In [ ]:
import json
import platform
import subprocess
import sys

print("Notebook Python:", platform.python_version())
if platform.system() != "Linux":
    raise RuntimeError("This notebook targets a Linux Colab GPU runtime.")
VENV = PROJECT / ".venv-colab"
subprocess.run([
    sys.executable, str(PROJECT / "scripts/colab_runtime.py"),
    "setup", "--venv", str(VENV),
], cwd=PROJECT, check=True)
PYTHON = VENV / "bin/python"
if not PYTHON.is_file():
    raise RuntimeError("The isolated inference environment was not created.")
print("Inference environment:", PYTHON)

## Evaluate the actual inputs sequentially

Four separate processes test the reported striped garment, another garment,
another person, and the supported shirt category. Processes run sequentially
to release GPU memory between cases. Dresses and trousers remain unsupported
by this project's 2D adapter. Inputs and every saved intermediate stay under
this notebook's `/content/avatar-vton-...` directory.

Each input checksum is checked against the embedded source manifest. A model
error or quality rejection is retained in the report and does not turn into
an overlay. A completed inference is still **pending visual review**.

In [ ]:
import hashlib
import time
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUNS = PROJECT / "experiments/colab-runs" / RUN_ID
RUNS.mkdir(parents=True, exist_ok=True)
CASES = [
    ("reported-striped-shirt", "data/raw/studio-person.jpg",
     "data/raw/garment-demo.png", "t-shirt"),
    ("different-garment", "data/raw/studio-person.jpg",
     "data/raw/vton-official/garment-number-95.jpg", "t-shirt"),
    ("different-person", "data/raw/vton-official/person-model-5.png",
     "data/raw/vton-official/garment-rhude.jpg", "t-shirt"),
    ("shirt-category", "data/raw/studio-person.jpg",
     "data/raw/vton-official/garment-rhude.jpg", "shirt"),
]
CASE_RECORDS = []
for name, person, garment, category in CASES:
    for relative in (person, garment):
        actual = hashlib.sha256((PROJECT / relative).read_bytes()).hexdigest()
        if actual != SOURCE_MANIFEST["files"][relative]["sha256"]:
            raise RuntimeError(f"Fixture checksum changed: {relative}")
    destination = RUNS / name
    print(f"\nRunning {name}: {category}", flush=True)
    start = time.monotonic()
    completed = subprocess.run([
        str(PYTHON), str(PROJECT / "scripts/colab_runtime.py"), "run",
        "--person", str(PROJECT / person),
        "--garment", str(PROJECT / garment),
        "--category", category, "--output", str(destination),
    ], cwd=PROJECT, check=False)
    record = {
        "case": name, "person": person, "garment": garment,
        "category": category, "returnCode": completed.returncode,
        "seconds": round(time.monotonic() - start, 3),
        "visualAcceptance": "pending visual review",
    }
    CASE_RECORDS.append(record)
    (RUNS / "notebook-cases.json").write_text(
        json.dumps({"sourceSha256": SOURCE_SHA256, "cases": CASE_RECORDS}, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(record), flush=True)

In [ ]:
from IPython.display import Markdown, display
from PIL import Image

def show_image(path, caption):
    display(Markdown(caption))
    with Image.open(path) as original:
        preview = original.convert("RGB")
        preview.thumbnail((768, 1024))
        display(preview)

for record in CASE_RECORDS:
    name = record["case"]
    destination = RUNS / name
    display(Markdown(f"### {name}"))
    show_image(PROJECT / record["person"], "**Original person**")
    show_image(PROJECT / record["garment"], "**Actual garment conditioning image**")
    report_path = destination / "report.json"
    if report_path.is_file():
        report = json.loads(report_path.read_text(encoding="utf-8"))
        print(json.dumps(report, indent=2))
    output = destination / "tryon-2d.png"
    if record["returnCode"] == 0 and output.is_file():
        show_image(output, "**Generated candidate — pending visual review, not validated fitting**")
    else:
        display(Markdown("**FAILED / REJECTED — no successful try-on result.**"))
        diagnostic = destination / "vton-model-output.png"
        if diagnostic.is_file():
            show_image(diagnostic, "**Raw diagnostic only — not a final try-on result**")
    display(Markdown(
        "Inspect uploaded pattern/colors, neckline, shoulders, both sleeves and hem; "
        "look for any old-shirt fragments, missing skin, halo or seams. Compare the "
        "face/hair/hands and background with the original. Do not mark acceptance "
        "from an API exit code or an automated stripe check alone."
    ))

In [ ]:
import zipfile
from google.colab import files

RESULTS_ZIP = PROJECT / "colab-tryon-results.zip"
allowed_suffixes = {".png", ".jpg", ".jpeg", ".json", ".txt", ".log"}
with zipfile.ZipFile(RESULTS_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUNS.rglob("*")):
        if (path.is_file() and not path.is_symlink()
                and path.resolve().is_relative_to(RUNS.resolve())
                and path.suffix.lower() in allowed_suffixes):
            archive.write(path, path.relative_to(PROJECT))
    archive.write(PROJECT / "SOURCE_MANIFEST.json", "SOURCE_MANIFEST.json")
print("Actual run artifacts:", RESULTS_ZIP, RESULTS_ZIP.stat().st_size, "bytes")
# Browser downloads can be missed if a tab disconnects before the
# cell finishes. Retain bounded previews in saved notebook outputs.
# These are explicitly previews, never replacements for full PNGs.
import base64
import hashlib
import io
from IPython.display import display
from PIL import Image

for report_path in sorted(RUNS.glob("*/report.json"))[:4]:
    saved_report = json.loads(report_path.read_text(encoding="utf-8"))
    accepted_by_pipeline = saved_report["status"] == "generated"
    image_path = report_path.parent / (
        "tryon-2d.png" if accepted_by_pipeline else "vton-model-output.png"
    )
    recovery_record = {
        "type": "avatar-vton-recovery-preview",
        "case": report_path.parent.name,
        "previewOnly": True,
        "qualityValidated": False,
        "imageKind": "unreviewed candidate" if accepted_by_pipeline else "rejected diagnostic",
        **{key: saved_report.get(key) for key in (
            "status", "error", "code", "checkpoint", "nativeResolution",
            "inputSha256", "seconds",
        )},
    }
    if image_path.is_file():
        recovery_record["fullImageSha256"] = hashlib.sha256(image_path.read_bytes()).hexdigest()
    display({"application/json": recovery_record}, raw=True)
    if image_path.is_file():
        with Image.open(image_path) as original:
            preview = original.convert("RGB")
        preview.thumbnail((640, 640))
        while True:
            jpeg = io.BytesIO()
            preview.save(jpeg, format="JPEG", quality=80)
            if jpeg.tell() <= 128 * 1024:
                break
            preview.thumbnail((max(1, preview.width // 2), max(1, preview.height // 2)))
        display({"image/jpeg": base64.b64encode(jpeg.getvalue()).decode("ascii")}, raw=True)
files.download(str(RESULTS_ZIP))

## Optional result persistence

The next cell is disabled by default. Enabling it requests Google Drive
authorization and copies **only the result ZIP** to `MyDrive/AvatarVTON`.
It does not enumerate, read or modify any unrelated Drive files. Checkpoints
are not copied. Standard browser download above does not require Drive access.

In [ ]:
SAVE_RESULTS_TO_DRIVE = False
if SAVE_RESULTS_TO_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    drive_output = Path("/content/drive/MyDrive/AvatarVTON")
    drive_output.mkdir(parents=True, exist_ok=True)
    target = drive_output / ("results-" + SOURCE_SHA256[:12] + ".zip")
    shutil.copy2(RESULTS_ZIP, target)
    print("Saved this run's result archive:", target)
else:
    print("Drive persistence disabled; no Drive access requested.")